# Laya smoke test + Banking77 check (Colab)

**Before running: Runtime -> Change runtime type -> T4 GPU -> Save.**
Then Runtime -> Run all.

Two things happen here:

1. **Step 1** runs one real Laya prediction through the whole harness. Everything in
   `src/runners/laya_runner.py` was written by reading Laya's own `rl_agent_api.py`
   and has never touched the real package. This is where we find out if that reading
   was right.
2. **Step 2** runs Banking77 (77 labels) and compares against the 0.425 the model card
   claims. If we land near it, the harness is wired correctly and the k-sweep can run.

In [ ]:
import platform
print("python", platform.python_version())
!nvidia-smi -L

In [ ]:
%cd /content
!rm -rf /content/HARNESS
!git clone --depth 1 https://github.com/Vansh-kap-98/HARNESS.git /content/HARNESS
%cd /content/HARNESS
!pip install -q laya datasets

In [ ]:
import sys
sys.path.insert(0, "/content/HARNESS")

import laya
print("laya", getattr(laya, "__version__", "no __version__"))

!cd /content/HARNESS && python -m pytest -q

## Step 1: one real Laya call

In [ ]:
import json, time
from laya import Router
from src.compiler import compile_schema_file

router = Router(preload=True, device="cuda")

SCHEMA = "/content/HARNESS/schemas/support_ticket.json"
compiled = compile_schema_file(SCHEMA)

TICKET = ("Hi, we were billed twice for March. Please refund the duplicate today "
          "or we will cancel our plan and move to a competitor.")

# compiled.questions is already in Laya's question format; nothing is reshaped.
result = router.predict(TICKET, compiled.questions)
print(json.dumps(result, indent=2, default=str)[:4000])

In [ ]:
from src.assembler import assemble, escalate_fields
from src.runners.laya_runner import reported_confidence, to_answers
from src.validate import validate_with_python_jsonschema

answers = to_answers(result, compiled)      # raises loudly if our reading was wrong
assembled = assemble(compiled, answers)

print(assembled.to_json())
print("checkpoint used     :", result["routing"]["model"])
print("escalate below 0.70 :", escalate_fields(assembled, 0.70))
print("laya's own conf     :", reported_confidence(result, compiled))
print("valid (dev engine)  :", validate_with_python_jsonschema(SCHEMA, assembled.instance).valid)

## Step 2: Banking77 at k=77

A neighbourhood check, not a reproduction: we do not know which split the card used,
what criteria text it gave each label, or whether it was zero-shot. We use the test
split, the English checkpoint, and the label names as their own criteria text.
Roughly 0.35-0.50 means the harness agrees with the card.

In [ ]:
from datasets import load_dataset

ds = load_dataset("PolyAI/banking77", split="test")
LABELS = list(ds.features["label"].names)
print(len(LABELS), "labels,", len(ds), "test queries")
print("first five:", LABELS[:5])

In [ ]:
from src.sweep import approx_tokens_by_chars

# Banking77 ships no descriptions, so each label is its own criteria text.
BANKING_INSTRUCTIONS = "Which customer-service intent does this message express?"
criteria = {name: name.replace("_", " ") for name in LABELS}
questions = {"intent": {"type": "choice",
                        "instructions": BANKING_INSTRUCTIONS,
                        "criteria": criteria}}

budget = approx_tokens_by_chars(
    BANKING_INSTRUCTIONS + "".join(criteria) + "".join(criteria.values()))
print("options alone ~%d tokens (english checkpoint context is 512)" % budget)

In [ ]:
LIMIT = 500        # quick first pass; set to None for the full test split

rows = ds if LIMIT is None else ds.select(range(LIMIT))
predictions, confidences = [], []
start = time.time()

for index, row in enumerate(rows):
    answer = router.predict(row["text"], questions, model="english")["answers"]["intent"]
    predictions.append(answer["choice"])
    confidences.append(max(answer["probabilities"]))
    if (index + 1) % 100 == 0:
        print("%d/%d  %.0fs" % (index + 1, len(rows), time.time() - start))

elapsed = time.time() - start
print("done: %d queries in %.0fs (%.1f ms each)" % (len(rows), elapsed, 1000 * elapsed / len(rows)))

In [ ]:
from src.score import Proportion, calibration

gold = [LABELS[row["label"]] for row in rows]
hits = sum(1 for expected, got in zip(gold, predictions) if expected == got)
accuracy = Proportion(hits, len(gold))

print("accuracy   :", accuracy)
print("chance     : %.4f  (1/%d)" % (1.0 / len(LABELS), len(LABELS)))
print("card claims: 0.425")

correct = [expected == got for expected, got in zip(gold, predictions)]
cal = calibration(confidences, correct)
print("ECE        : %.3f  (reportable: %s)" % (cal.ece, cal.reportable))

if 0.35 <= accuracy.value <= 0.50:
    print("
PIPELINE AGREES with the card. Proceed to the k sweep.")
else:
    print("
OUT OF NEIGHBOURHOOD. Do not run the sweep yet -- find out why first.")
    print("Check: criteria text, checkpoint, whether every option fit the context.")